# `_check_.ipynb` — inspect **`src_1/_dta_raw_fetched.pkl`**

Per **stock**: calendar date span, **missing** counts (NaN) for OHLCV, and **Close** min/max (**range**).

The first cell runs `_my_builder_1/auto.py` via `runpy` (same `sys.path` fix as `python auto.py`, without running `main()`). If imports fail, **Kernel → Restart** and run this cell again. Read-only — does not refetch Yahoo.

In [1]:
import os
import runpy
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _bootstrap_src1_via_auto() -> None:
    """Run `auto.py` that sits next to `src_1/` (not repo-root `auto.py` for src_2)."""
    mark = "auto.py"
    here = Path.cwd().resolve()

    def is_src1_builder_auto(p: Path) -> bool:
        return (
            p.is_file()
            and p.name == mark
            and (p.parent / "src_1" / "__init__.py").is_file()
        )

    def try_run(p: Path) -> bool:
        if not is_src1_builder_auto(p):
            return False
        runpy.run_path(str(p), run_name="src1_notebook_bootstrap")
        return True

    for base in [here, *here.parents][:45]:
        if try_run(base / mark):
            return
        try:
            for ch in sorted(base.iterdir()):
                if ch.is_dir() and try_run(ch / mark):
                    return
        except OSError:
            continue

    for dirpath, dirnames, filenames in os.walk(str(here)):
        depth = len(Path(dirpath).relative_to(here).parts)
        if depth > 12:
            dirnames[:] = []
            continue
        if mark in filenames:
            p = Path(dirpath) / mark
            if try_run(p):
                return

    raise FileNotFoundError(
        "Could not find auto.py next to src_1/ (e.g. _my_builder_1/auto.py). "
        "Cd to this repo, then Kernel → Restart and run this cell again."
    )


_bootstrap_src1_via_auto()

from src_1._0_fns_get_data import default_raw_pkl_path

PKL = default_raw_pkl_path()
df = pd.read_pickle(PKL)
print("Loaded:", PKL)
print("Panel shape:", df.shape, "| index:", df.index.min(), "..", df.index.max())

Loaded: /Users/yerik/_apple_lib/_peg_ProgEnvGit/a4yy_STRATEGY/builders/_my_builder_1/src_1/_dta_raw_fetched.pkl
Panel shape: (1827, 150) | index: 2021-04-18 00:00:00 .. 2026-04-18 00:00:00


### Per-stock table: dates, missing values, Close range

In [2]:
_FIELDS = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]


def panel_quality_table(df: pd.DataFrame) -> pd.DataFrame:
    """One row per ticker: date range, NaN counts/pcts, Close min/max."""
    if df.empty or not isinstance(df.columns, pd.MultiIndex):
        return pd.DataFrame()
    syms = sorted(df.columns.get_level_values(0).unique())
    rows: list[dict] = []
    n_idx = len(df.index)
    for sym in syms:
        block = df[sym] if sym in df.columns.get_level_values(0) else None
        if block is None or block.empty:
            continue
        close = block["Close"] if "Close" in block.columns else pd.Series(dtype=float)
        vclose = close.dropna()
        na_close = int(close.isna().sum())
        row: dict = {
            "symbol": sym,
            "idx_date_start": df.index.min(),
            "idx_date_end": df.index.max(),
            "close_first_date": vclose.index.min() if len(vclose) else pd.NaT,
            "close_last_date": vclose.index.max() if len(vclose) else pd.NaT,
            "rows_calendar": n_idx,
            "close_valid_rows": int(vclose.shape[0]),
            "close_na_count": na_close,
            "close_na_pct": round(100.0 * na_close / n_idx, 2) if n_idx else np.nan,
            "close_min": float(vclose.min()) if len(vclose) else np.nan,
            "close_max": float(vclose.max()) if len(vclose) else np.nan,
        }
        for f in _FIELDS:
            if f == "Close":
                continue  # already in close_* columns above
            if f not in block.columns:
                row[f"{f}_na"] = np.nan
                continue
            s = block[f]
            row[f"{f}_na"] = int(s.isna().sum())
            row[f"{f}_na_pct"] = round(100.0 * s.isna().sum() / n_idx, 2) if n_idx else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


summary = panel_quality_table(df)
# compact view first
cols_show = [
    "symbol",
    "close_first_date",
    "close_last_date",
    "close_valid_rows",
    "close_na_pct",
    "close_min",
    "close_max",
]
try:
    display(summary[cols_show])
except NameError:
    print(summary[cols_show].to_string(index=False))

,symbol,close_first_date,close_last_date,close_valid_rows,close_na_pct,close_min,close_max
0,AAPL,2021-04-19,2026-04-18,1826,0.05,122.769997,286.190002
1,ACHR,2021-04-19,2026-04-18,1826,0.05,1.630000,13.640000
2,AMD,2021-04-19,2026-04-18,1826,0.05,55.939999,278.390015
3,ANDE,2021-04-19,2026-04-18,1826,0.05,25.610001,74.699997
4,APPL,NaT,NaT,0,100.00,NaN,NaN
5,ASML,2021-04-19,2026-04-18,1826,0.05,379.130005,1526.510010
6,AVGO,2021-04-19,2026-04-18,1826,0.05,42.237999,412.970001
7,COIN,2021-04-19,2026-04-18,1826,0.05,32.529999,419.779999
8,CRSP,2021-04-19,2026-04-18,1826,0.05,31.270000,161.889999
9,CURSOR,NaT,NaT,0,100.00,NaN,NaN


### Full detail: NaN counts for every OHLCV field

In [3]:
na_cols = [c for c in summary.columns if c.endswith("_na") and not c.endswith("_na_pct")]
pct_cols = [c for c in summary.columns if c.endswith("_na_pct")]
detail = summary[["symbol"] + na_cols + pct_cols]
try:
    display(detail)
except NameError:
    print(detail.to_string(index=False))

,symbol,Open_na,High_na,Low_na,Adj Close_na,Volume_na,close_na_pct,Open_na_pct,High_na_pct,Low_na_pct,Adj Close_na_pct,Volume_na_pct
0,AAPL,1,1,1,1,1,0.05,0.05,0.05,0.05,0.05,0.05
1,ACHR,1,1,1,1,1,0.05,0.05,0.05,0.05,0.05,0.05
2,AMD,1,1,1,1,1,0.05,0.05,0.05,0.05,0.05,0.05
3,ANDE,1,1,1,1,1,0.05,0.05,0.05,0.05,0.05,0.05
4,APPL,1827,1827,1827,1827,1827,100.00,100.00,100.00,100.00,100.00,100.00
5,ASML,1,1,1,1,1,0.05,0.05,0.05,0.05,0.05,0.05
6,AVGO,1,1,1,1,1,0.05,0.05,0.05,0.05,0.05,0.05
7,COIN,1,1,1,1,1,0.05,0.05,0.05,0.05,0.05,0.05
8,CRSP,1,1,1,1,1,0.05,0.05,0.05,0.05,0.05,0.05
9,CURSOR,1827,1827,1827,1827,1827,100.00,100.00,100.00,100.00,100.00,100.00
